In [1]:
from get_data.get import *
from calculate.calculate import *
from visualize.visualize import *
from db.db import *
from db.load import *

2026-04-01 17:37:03.470 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


In [2]:
# Get data building data
df_grid, grid_bd_points = get_latest_grid_data()
dfs1 = get_dfs1()

# Get pop/density data
dfs2 = get_dfs2(df_grid)

# Set score
weight_dic = {'broadcast': 0.41787,
             'electricity':0.02198,
             'factory':0.00376,
             'hospital':0.00064,
             'infra':0.00075,
             'prison':0.02757,
             'public':0.01237,
             'science':0.08148,
             'telecommunication':0.14754,
             'transportation':0.00187,
             'water':0.17237,
             'frequency':0}
set_score(dfs1, weight_dic)

# Parameters
RANGE_KM = 2.0

# Rank calculation
rank_dic, max_radar_num = calc_rank(dfs1, df_grid, RANGE_KM, radar_num=50, polygon_coords=grid_bd_points)

df_population = dfs2['population']
df_area_density = dfs2['area_density']

df_final = get_df_final(rank_dic, df_grid, df_population, df_area_density, RANGE_KM)

upload_result(df_final)

2026-04-01 14:06:44.505 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-04-01 14:06:44.507 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


1/2단계: DB 연결 및 체크 성공


d:\workspaces\00_Project\crawlProject\test_project\project\get_data\get.py:103: UserWarning: Geometry column does not contain geometry.
  gdf['geometry'] = gdf['geometry'].apply(lambda x: str(x))
2026-04-01 14:06:46.995 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


------------------------------
1순위
위치 : [ 37.528251 126.965614]
점수 : 0.00563
남은 시설물 : 0개
------------------------------
최대 radar 개수: 1
저장 완료: case6


In [3]:
# Icon config
ICON_MAP = {
    "broadcast":         folium.Icon(color="orange",     icon="broadcast-tower",   prefix="fa"),
    "electricity":       folium.Icon(color="green",      icon="bolt",              prefix="fa"),
    "factory":           folium.Icon(color="blue",       icon="industry",          prefix="fa"),
    "hospital":          folium.Icon(color="red",        icon="hospital",          prefix="fa"),
    "infra":             folium.Icon(color="darkblue",   icon="cogs",              prefix="fa"),
    "prison":            folium.Icon(color="black",      icon="university",        prefix="fa"),
    "public":            folium.Icon(color="cadetblue",  icon="building",          prefix="fa"),
    "science":           folium.Icon(color="pink",       icon="flask",             prefix="fa"),
    "telecommunication": folium.Icon(color="beige",      icon="satellite-dish",    prefix="fa"),
    "transportation":    folium.Icon(color="darkgreen",  icon="train",             prefix="fa"),
    "water":             folium.Icon(color="lightblue",  icon="tint",              prefix="fa"),
    "frequency":         folium.Icon(color="darkred",    icon="signal",            prefix="fa"),
}
visualize(df_grid, dfs1, rank_dic, RANGE_KM, ICON_MAP,
              show_rank=None, polygon_coords=grid_bd_points,
              df_final=df_final)

저장 완료: map.html


In [9]:
delete_result('case2')

#### Setting 함수 test

In [72]:
import subprocess

def run_cmd(commands):
    # 문자열이면 그대로, 리스트면 && 로 연결
    full_command = commands if isinstance(commands, str) else " && ".join(commands)
    result = subprocess.run(
        full_command,
        shell=True,
        capture_output=True,
        text=True,
        encoding='cp949'
    )
    if result.returncode != 0:
        print(f"❌ 오류: {result.stderr}")
    return result

In [123]:
def check_env():
    result = run_cmd('conda env list')
    str_list = result.stdout.strip().split('\n')

    env_list = []
    for line in str_list:
        if line == '' or line.startswith('#'):
            continue
        else:
            env_list.append(line.split()[0])
        
    return env_list

In [124]:
check_env()

['base', 'jsenv', 'test']

In [125]:
def create_env(env_name, python_version='3.10'):
    
    # 이미 존재하는 환경인지 확인
    existing_envs = check_env()
    
    if env_name in existing_envs:
        print(f"'{env_name}' 환경이 이미 존재합니다.")
        return

    # 환경 생성
    print(f"'{env_name}' 환경 생성 중...")
    run_cmd(f'conda create -n {env_name} python={python_version} -y')
    print(f"'{env_name}' 환경 생성 완료")

In [126]:
create_env('test2',python_version='3.14')

'test2' 환경 생성 중...
'test2' 환경 생성 완료


In [127]:
def delete_env(env_name):
    """
    conda 가상환경 삭제

    Parameters
    ----------
    env_name : 삭제할 환경 이름
    """
    # 이미 존재하는 환경인지 확인
    existing_envs = check_env()

    if env_name not in existing_envs:
        print(f"'{env_name}' 환경이 존재하지 않습니다.")
        return

    # base 환경 삭제 방지
    if env_name == 'base':
        print("base 환경은 삭제할 수 없습니다.")
        return

    # 환경 삭제
    print(f"'{env_name}' 환경 삭제 중...")
    run_cmd(f'conda env remove -n {env_name} -y')
    print(f"'{env_name}' 환경 삭제 완료")

In [134]:
# 사용
delete_env('test2')

'test2' 환경 삭제 중...
'test2' 환경 삭제 완료


In [128]:
def check_packages(env_name):
    result = run_cmd(f'conda run -n {env_name} pip list')
    
    packages = []
    lines = [line for line in result.stdout.split('\n')[2:] if line.strip()]
    for line in lines:
        packages.append(line.split()[0])

    return packages

In [129]:
def install_packages(env_name, packages: list):
    existing_packages = check_packages(env_name)

    already = [pkg for pkg in packages if pkg in existing_packages]
    to_install = [pkg for pkg in packages if pkg not in existing_packages]

    for pkg in already:
        print(f"'{pkg}'패키지가 이미 존재합니다.")

    if not to_install:
        print("모든 패키지가 이미 설치되어 있습니다.")
        return

    pkg_str = ' '.join(to_install)
    print(f"'{env_name}' 환경에 {pkg_str} 설치 중...")
    run_cmd(f'conda run -n {env_name} pip install {pkg_str}')
    print("설치 완료")

In [130]:
check_packages('test2')

['pip']

In [131]:
install_packages('test2',['numpy','pandas','scipy', 'folium'])

'test2' 환경에 numpy pandas scipy folium 설치 중...
설치 완료


In [132]:
def uninstall_packages(env_name, packages: list):
    """
    지정한 환경에서 패키지 삭제

    Parameters
    ----------
    env_name : 삭제할 환경 이름
    packages : 삭제할 패키지 리스트
    """
    existing_packages = check_packages(env_name)

    # 설치된 것과 아닌 것 분리
    not_found  = [pkg for pkg in packages if pkg not in existing_packages]
    to_remove  = [pkg for pkg in packages if pkg in existing_packages]

    for pkg in not_found:
        print(f"'{pkg}' 패키지가 존재하지 않습니다.")

    if not to_remove:
        print("삭제할 패키지가 없습니다.")
        return

    pkg_str = ' '.join(to_remove)
    print(f"'{env_name}' 환경에서 {pkg_str} 삭제 중...")
    run_cmd(f'conda run -n {env_name} pip uninstall {pkg_str} -y')
    print("삭제 완료")

In [133]:
uninstall_packages('test2', ['pandas', 'numpy'])

'test2' 환경에서 pandas numpy 삭제 중...
삭제 완료


#### SQL 서버 테스트

In [6]:
import pandas as pd
from sqlalchemy import create_engine
import urllib

# ── 1. 데이터베이스 연결 설정 ──────────────────────────────
# SSMS에서 접속했던 서버 이름과 방금 만든 데이터베이스 이름을 입력한다.
server = r'localhost\SQLEXPRESS'  # 본인의 서버 이름 (예: 'localhost' 또는 '.')
database = 'test'       # 접속할 데이터베이스 이름

# Windows 인증(Trusted_Connection) 및 인증서 신뢰(TrustServerCertificate) 옵션 필수 포함
params = urllib.parse.quote_plus(
    f'DRIVER={{ODBC Driver 17 for SQL Server}};'
    f'SERVER={server};'
    f'DATABASE={database};'
    f'Trusted_Connection=yes;'
    f'TrustServerCertificate=yes;'
)

# SQLAlchemy 엔진 생성 (파이썬과 SQL 서버를 이어주는 다리 역할)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")


# ── 2. CSV 데이터를 SQL 서버로 적재 (INSERT) ─────────────
# 내 PC에 있는 CSV 파일을 pandas 데이터프레임으로 읽어온다.
csv_file_path = 'data/broadcast/df_broadcast.csv'  # 실제 CSV 파일 경로로 변경할 것
df = pd.read_csv(csv_file_path)

# 데이터프레임을 통째로 SQL 서버의 'MyDataTable'이라는 테이블로 밀어넣는다.
# if_exists='replace': 기존 테이블이 있으면 지우고 새로 덮어쓴다.
# if_exists='append': 기존 테이블 데이터 아래에 이어서 추가한다.
df.to_sql('broadcast', con=engine, if_exists='replace', index=False)
print("✅ CSV 데이터 SQL 서버 적재 완료!")


# ── 3. SQL 서버에서 데이터를 파이썬으로 가져오기 (SELECT) ──
# 적재된 데이터를 다시 파이썬으로 불러와서 확인해본다.
query = "SELECT * FROM broadcast"
df_loaded = pd.read_sql(query, con=engine)

print("\n📊 SQL 서버에서 불러온 데이터 미리보기:")
print(df_loaded.head())

✅ CSV 데이터 SQL 서버 적재 완료!

📊 SQL 서버에서 불러온 데이터 미리보기:
          name   latitude   longitude  tag  score
0       한국방송공사  37.525970  126.916717  방송국      1
1        ㈜문화방송  37.581123  126.890989  방송국      1
2       ㈜에스비에스  37.529190  126.873747  방송국      1
3      (재)국악방송  37.580854  126.889332  방송국      1
4  (재)국제방송교류재단  37.479127  127.008047  방송국      1


c:\Users\user2\miniconda3\envs\jsenv\Lib\site-packages\pandas\io\sql.py:1648: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


In [2]:
# Get data building data
df_grid, grid_bd_points = get_latest_grid_data()
dfs1 = get_dfs1_server()

# Get pop/density data
dfs2 = get_dfs2_server(df_grid)

# Set score
weight_dic = {'broadcast': 0.41787,
             'electricity':0.02198,
             'factory':0.00376,
             'hospital':0.00064,
             'infra':0.00075,
             'prison':0.02757,
             'public':0.01237,
             'science':0.08148,
             'telecommunication':0.14754,
             'transportation':0.00187,
             'water':0.17237,
             'frequency':0}
set_score(dfs1, weight_dic)

# Parameters
RANGE_KM = 2.0

# Rank calculation
rank_dic, max_radar_num = calc_rank(dfs1, df_grid, RANGE_KM, radar_num=50, polygon_coords=grid_bd_points)

df_population = dfs2['population']
df_area_density = dfs2['area_density']

df_final = get_df_final(rank_dic, df_grid, df_population, df_area_density, RANGE_KM)


d:\workspaces\00_Project\crawlProject\test_project\project\get_data\get.py:64: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  with temp_engine.connect() as conn:
2026-04-01 16:42:50.131 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


🔄 기존 데이터베이스(projectdb) 삭제 중...
🆕 새 데이터베이스(projectdb) 생성 중...
✅ 'projectdb' 초기화 및 생성 완료!


d:\workspaces\00_Project\crawlProject\test_project\project\get_data\get.py:132: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  with engine.connect() as conn:


1/2단계: DB 연결 및 체크 성공


d:\workspaces\00_Project\crawlProject\test_project\project\get_data\get.py:169: UserWarning: Geometry column does not contain geometry.
  gdf['geometry'] = gdf['geometry'].apply(lambda x: str(x))


------------------------------
1순위
위치 : [ 37.528251 126.965614]
점수 : 0.00563
남은 시설물 : 0개
------------------------------
최대 radar 개수: 1


## FINAL

In [ ]:
data_list1 = ['broadcast',
             'electricity',
             'factory',
             'hospital',
             'infra',
             'prison',
             'public',
             'science',
             'telecommunication',
             'transportation',
             'water',
             'frequency']

data_list2 = ['population_raw','density']

dfs1 = load_data(data_list1)
dfs2 = load_data(data_list2)

# Get data building data
df_grid, grid_bd_points = get_latest_grid_data()

# Set score
weight_dic = {'broadcast': 0.41787,
             'electricity':0.02198,
             'factory':0.00376,
             'hospital':0.00064,
             'infra':0.00075,
             'prison':0.02757,
             'public':0.01237,
             'science':0.08148,
             'telecommunication':0.14754,
             'transportation':0.00187,
             'water':0.17237,
             'frequency':0}
set_score(dfs1, weight_dic)

# Parameters
RANGE_KM = 2.0

# Rank calculation
rank_dic, max_radar_num = calc_rank(dfs1, df_grid, RANGE_KM, radar_num=50, polygon_coords=grid_bd_points)

df_population = get_df_population(dfs2['population_raw'], df_grid)
df_area_density = get_df_area_density(dfs2['density'], df_grid)

df_final = get_df_final(rank_dic, df_grid, df_population, df_area_density, RANGE_KM)

upload_result_server(df_final)

c:\Users\user2\miniconda3\envs\jsenv\Lib\site-packages\pandas\io\sql.py:1648: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())
